# 04 · Modelling (baseline)
### BNI Ecosystem & Growth Intelligence — Kelompok 8 · ODP BNI Data Analytics Batch 2

| # | Notebook | Isi |
|---|---|---|
| 01 | Dataset Generation & Validation | data sintetis v8 + validasi |
| 02 | EDA | temuan, hipotesis, kontrak fitur |
| 03 | Feature Engineering | split, dataset fitur final, uji nilai fitur |
| **04** | **Modelling** | **notebook ini — model risiko M0–M4, prediksi OOF, gerbang promosi v0, aturan anomali + 1 challenger, skor v0** |
| 05 | Modelling Optimization | HPO, ensemble, SMOTE vs bobot, kalibrasi, conformal, ambang |
| 06 | Model Evaluation | evaluasi **sekali** di test & ground truth |
| 07 | Post-evaluation & Remodelling | analisis kesalahan |

## Tujuan (prioritas Hari 2, `MODELLING_PLAN_v8` §10)
1. **Klasifikasi risiko** Low/Medium/High dengan `set_core` (40 fitur):
   - M0 dummy (lantai) dan **M1 logistic regression** (champion awal);
   - challenger M2 Random Forest, M3 XGBoost, M4 LightGBM — semuanya **versi ordinal monoton** (§5.1, §5.3).
2. **Prediksi out-of-fold (OOF)** 5-fold `cv_fold` untuk semua model, plus prediksi test & tanpa label (disimpan, **tidak dinilai**).
3. **Kerangka sistem keputusan 4 lapis versi 0**: kurva risk–coverage, conformal LAC sederhana, keputusan Otomatis / Review, dan **gerbang promosi 4 kriteria** (champion vs challenger).
4. **Deteksi anomali "Why Now"**: champion = aturan transparan per jenis kejadian, challenger = skor gabungan rank-average untuk distress & ekspansi; challenger 2 = Isolation Forest online (pembanding).
5. **Skor v0 untuk dashboard**: `outputs/scores/risk_scores.csv` dan `outputs/scores/anomaly_alerts.csv`.

## Aturan yang dijaga (checklist §11)
- Label `risk_class` baris **test dikunci** (dihapus dari tabel kerja) sebelum apa pun dilatih. Test baru dinilai di NB06.
- `PeerZScaler` di-fit **di dalam setiap fold**. Kolom `*_pz` di file tidak dipakai langsung.
- `gt_*`, `eval_targets/` hanya untuk evaluasi; anomali dinilai **hanya di perusahaan non-test**.
- Tidak ada kolom proxy-label, `relationship_end`, atau `data_visibility` di matriks fitur.

## Input & output (relatif ke akar proyek)
| | Path | Asal / pemakai |
|---|---|---|
| **input** | `src/features.py` | modul NB03: `PeerZScaler`, `load_features`, `load_feature_sets` |
| **input** | `data/processed/` | tabel fitur risiko & panel anomali dari NB03 |
| **input** | `data/processed/feature_sets.json` | daftar set fitur NB03 (NB04 menambahkan `set_noncredit` & `set_prospect`) |
| **input** | `data/processed/eval_targets/anomaly_events_monthly.csv` | ground truth anomali, **hanya dibaca untuk perusahaan non-test** |
| **output** | `outputs/04_modelling/*.csv`, `SUMMARY.md` | hasil OOF, uji logika, gerbang promosi, registri model, evaluasi anomali |
| **output** | `outputs/04_modelling/figures/*.png` | grafik E1, G1, J1 |
| **output** | `outputs/04_modelling/predictions/` | prediksi semua model (OOF, test, tanpa label) |
| **output** | `models/04_modelling/` | model final terlatih (`*.joblib`), `manifest.json` (champion, set fitur, ambang), `model_files.csv` |
| **output** | `outputs/scores/` | `risk_scores.csv`, `anomaly_alerts.csv`, `shadow_log.csv` untuk dashboard |

Jalankan NB03 dulu. Notebook ini mencari akar proyek (folder berisi `pyproject.toml`), jadi boleh dijalankan dari folder `notebooks/`. Waktu jalan ±2–5 menit.

**Daftar isi:** A. Setup · B. Data & pagar kebocoran · C. Model · D. Train → valid · E. Prediksi OOF · F. Uji logika bisnis · G. Lapis 2–3 v0 · H. Gerbang promosi v0 · I. Skor risiko v0 · J. Anomali · K. Ringkasan & checklist · Daftar pustaka

## A. Setup
XGBoost dan LightGBM dipasang lewat `uv sync` (ada di `pyproject.toml`). Bila salah satunya tidak terpasang, notebook memakai pengganti dari scikit-learn dengan protokol yang sama dan **mencatat penggantinya** di registri model:
- XGBoost → `HistGradientBoosting` tumbuh per kedalaman (`max_depth=3`);
- LightGBM → `HistGradientBoosting` tumbuh per daun (`max_leaf_nodes=15`, `max_features=0.8`).

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import joblib
import sklearn
from threadpoolctl import threadpool_limits
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, roc_auc_score, log_loss, confusion_matrix
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

try:
    import xgboost as xgb; HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    import lightgbm as lgb; HAS_LGB = True
except ImportError:
    HAS_LGB = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
threadpool_limits(1)          # data kecil: 1 thread BLAS/OpenMP lebih cepat daripada banyak thread yang berebut (sama dengan NB03)

SEED = 2026
T0 = time.time()

In [ ]:
def find_root(start=Path.cwd()):
    """Akar proyek = folder terdekat (dari cwd ke atas) yang berisi pyproject.toml."""
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists(): return p
    raise FileNotFoundError('pyproject.toml tidak ditemukan — jalankan notebook dari dalam folder proyek')

PROJECT_ROOT = find_root()
SRC    = PROJECT_ROOT / 'src'
PROC   = PROJECT_ROOT / 'data' / 'processed'          # INPUT: output NB03
EVAL   = PROC / 'eval_targets'                        # INPUT: ground truth, hanya untuk evaluasi
OUT    = PROJECT_ROOT / 'outputs' / '04_modelling'    # OUTPUT: tabel evaluasi, registri, ringkasan
FIG    = OUT / 'figures'
PRED   = OUT / 'predictions'
SCORES = PROJECT_ROOT / 'outputs' / 'scores'          # OUTPUT: skor v0 untuk dashboard
MODEL_DIR = PROJECT_ROOT / 'models' / '04_modelling'  # OUTPUT: model final terlatih (joblib) untuk NB05+

REQUIRED = [SRC / 'features.py', PROC / 'feature_sets.json', EVAL / 'anomaly_events_monthly.csv']
missing = [str(p.relative_to(PROJECT_ROOT)) for p in REQUIRED if not p.exists()]
if missing:
    raise FileNotFoundError('Output NB03 belum ada: ' + ', '.join(missing) + ' → jalankan notebooks/03_Feature_Engineering.ipynb dulu.')
for d in (OUT, FIG, PRED, SCORES, MODEL_DIR): d.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
from features import PeerZScaler, RATIO_COLS, PEER_KEYS, load_features, load_feature_sets
from modelling import MONO, OrdinalMonotone          # di modul supaya model joblib bisa di-load di notebook lain

print('PROJECT_ROOT:', PROJECT_ROOT)
print('XGBoost  :', xgb.__version__ if HAS_XGB else 'tidak ada → pengganti HGB per kedalaman')
print('LightGBM :', lgb.__version__ if HAS_LGB else 'tidak ada → pengganti HGB per daun')

In [ ]:
C = dict(s1='#2a78d6', s2='#eb6834', s3='#1baf7a', s4='#eda100', s5='#e87ba4', s6='#008300', s7='#6250d6', s8='#e34948',
         ink='#0b0b0b', ink2='#52514e', muted='#898781', grid='#e1e0d9', axis='#c3c2b7', surface='#fcfcfb')
mpl.rcParams.update({'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'axes.edgecolor': C['axis'], 'axes.linewidth': .8,
                     'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': .6, 'axes.axisbelow': True, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'axes.labelcolor': C['ink2'], 'axes.labelsize': 9, 'xtick.color': C['muted'], 'ytick.color': C['muted'],
                     'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.frameon': False, 'legend.fontsize': 8, 'lines.linewidth': 2,
                     'figure.dpi': 110, 'text.color': C['ink']})
def save_fig(fig, name): fig.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())

def write_table(df, path_no_ext):
    """parquet bila pyarrow ada, kalau tidak csv.gz."""
    try:
        df.to_parquet(f'{path_no_ext}.parquet', index=False); return Path(f'{path_no_ext}.parquet').name
    except Exception:
        df.to_csv(f'{path_no_ext}.csv.gz', index=False, compression='gzip'); return Path(f'{path_no_ext}.csv.gz').name

RISK_ORDER = ['Low', 'Medium', 'High']; Y_MAP = {k: i for i, k in enumerate(RISK_ORDER)}

## B. Data & pagar kebocoran
- **Set fitur**: `set_core` (40) untuk nasabah kredit. Dua turunan untuk populasi yang tidak punya kolom tertentu (§2.4):
  - `set_noncredit` (38) = `set_core` tanpa `utilization`, `log_limit` — nasabah non-kredit;
  - `set_prospect` = `set_core ∩ set_portable` — prospect (tanpa transaksi BNI).
- **Label test dikunci**: `risk_class` baris test dihapus dari tabel kerja. Label aslinya baru dibaca lagi di NB06.

In [ ]:
RISK_ALL = load_features(PROJECT_ROOT, 'risk')
fs = load_feature_sets(PROJECT_ROOT)
CORE = fs['set_core']
NONCREDIT = [c for c in CORE if c not in ('utilization', 'log_limit')]
PROSPECT = [c for c in CORE if c in fs['set_portable']]
ANOM = fs['anomaly']
FEATURE_SETS = {'set_core': CORE, 'set_noncredit': NONCREDIT, 'set_prospect': PROSPECT}

# pagar kolom
FORBIDDEN = {'collectibility', 'max_dpd_12m', 'watchlist_flag', 'restructuring_flag', 'relationship_end', 'data_visibility', 'risk_class'}
for name, cols in {**FEATURE_SETS, 'anomaly': ANOM}.items():
    bad = [c for c in cols if c in FORBIDDEN or c.startswith('gt_') or 'true_' in c]
    assert not bad, f'{name} memuat kolom terlarang: {bad}'

# kunci label test
df = RISK_ALL.copy()
n_test_labels = df.loc[df.split == 'test', 'risk_class'].notna().sum()
df.loc[df.split == 'test', 'risk_class'] = np.nan
del RISK_ALL
assert df.loc[df.split == 'test', 'risk_class'].isna().all()
print(f'label test dikunci: {n_test_labels} label dihapus dari tabel kerja')

# simpan set turunan ke feature_sets.json (pekerjaan tertunda NB03, §12.5)
fs_path = PROC / 'feature_sets.json'; fs_json = json.loads(fs_path.read_text(encoding='utf-8'))
if 'set_noncredit' not in fs_json or 'set_prospect' not in fs_json:
    fs_json['set_noncredit'] = NONCREDIT; fs_json['set_prospect'] = PROSPECT
    fs_json.setdefault('notes', {})['set_noncredit'] = 'set_core tanpa utilization & log_limit (nasabah non-kredit); ditambahkan NB04'
    fs_json['notes']['set_prospect'] = 'set_core ∩ set_portable (prospect); ditambahkan NB04'
    fs_path.write_text(json.dumps(fs_json, ensure_ascii=False, indent=1), encoding='utf-8')

DEV = df[df.split.isin(['train', 'valid'])].reset_index(drop=True)
y_dev = DEV.risk_class.map(Y_MAP).astype(int).values
TR_MASK = (DEV.split == 'train').values; VA_MASK = (DEV.split == 'valid').values
SCORE_POP = df[df.split.isin(['test', 'unlabeled'])].reset_index(drop=True)

summ = pd.DataFrame({'set': list(FEATURE_SETS), 'n_fitur': [len(v) for v in FEATURE_SETS.values()],
                     'dipakai untuk': ['nasabah kredit (train/valid/test)', 'nasabah non-kredit (400, tanpa label)', 'prospect (600, tanpa label)']})
display(summ)
display(pd.crosstab(df.split, df.customer_type).assign(**{'% High (berlabel)': df.groupby('split').risk_class.apply(lambda s: round(100 * (s == 'High').sum() / s.notna().sum(), 1) if s.notna().any() else np.nan)}))
print('kolom set_core yang dibuang untuk prospect:', [c for c in CORE if c not in PROSPECT])

## C. Model
### C1. Pipeline umum
Setiap model = `ColumnTransformer` (PeerZScaler di-fit ulang di dalam fold → `der_pz`, `cr_pz`, `npm_pz`, `icr_pz`; kolom lain diteruskan) → estimator. Kolom `*_pz` yang tersimpan di file **tidak dipakai**, karena statistiknya di-fit pada train + unlabeled.

### C2. Ordinal monoton untuk model pohon (Frank & Hall, 2001)
Dua model biner: P(y > Low) dan P(y > Medium). Probabilitas kelas:
- P(Low) = 1 − P(>Low);
- P(High) = P(>Medium), dipotong supaya ≤ P(>Low);
- P(Medium) = P(>Low) − P(>Medium).

Kedua model biner diberi **batasan monoton** yang sama (tabel §5.3). Bobot sampel = `balanced` per kelas asli (setara `class_weight='balanced'` di LR).

`MONO` dan `OrdinalMonotone` didefinisikan di `src/modelling.py`, bukan di notebook, supaya model yang disimpan (bagian I3) bisa di-load di NB05–07.


In [ ]:
PZ_OUT = list(PeerZScaler().get_feature_names_out())
print(f'batasan monoton: {sum(v > 0 for v in MONO.values())} fitur arah +, {sum(v < 0 for v in MONO.values())} fitur arah −, sisanya bebas')

def make_prep(cols):
    use_pz = any(c in PZ_OUT for c in cols); other = [c for c in cols if c not in PZ_OUT]
    parts = ([('pz', PeerZScaler(), RATIO_COLS + PEER_KEYS)] if use_pz else []) + [('rest', 'passthrough', other)]
    return ColumnTransformer(parts, remainder='drop', verbose_feature_names_out=False).set_output(transform='pandas')


BOOST_A = 'xgb' if HAS_XGB else 'hgb_depth'; BOOST_B = 'lgb' if HAS_LGB else 'hgb_leaf'
MODELS = {   # kode: (nama, keluarga, pengganti?)
    'M0': ('Dummy (kelas mayoritas)', 'lantai', ''),
    'M1': ('Logistic regression (scorecard)', 'linear', ''),
    'M2': ('Random Forest ordinal monoton', 'bagging', ''),
    'M3': ('XGBoost ordinal monoton', 'boosting', '' if HAS_XGB else 'HGB per kedalaman (XGBoost tidak terpasang)'),
    'M4': ('LightGBM ordinal monoton', 'boosting', '' if HAS_LGB else 'HGB per daun (LightGBM tidak terpasang)'),
    'M3x': ('Boosting multikelas TANPA batasan (pembanding)', 'boosting', 'HGB multikelas'),
}
LR_C = 0.5    # diganti hasil grid di bagian D

def make_model(code, cols, C_=None):
    if code == 'M0':
        est = DummyClassifier(strategy='most_frequent')
    elif code == 'M1':
        est = make_pipeline(SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True), StandardScaler(),
                            LogisticRegression(C=C_ or LR_C, class_weight='balanced', max_iter=5000))
    elif code == 'M2':
        est = OrdinalMonotone('rf')
    elif code == 'M3':
        est = OrdinalMonotone(BOOST_A)
    elif code == 'M4':
        est = OrdinalMonotone(BOOST_B)
    elif code == 'M3x':
        est = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=20, l2_regularization=1.0,
                                             class_weight='balanced', early_stopping=False, max_bins=63, random_state=SEED)
    return Pipeline([('prep', make_prep(cols)), ('est', est)])

def fit_model(code, cols, X, y, **kw):
    m = make_model(code, cols, **kw)
    if code in ('M2', 'M3', 'M4'):
        return m.fit(X, y, est__sample_weight=compute_sample_weight('balanced', y))
    return m.fit(X, y)

def metrics(y, P, prefix=''):
    pred = P.argmax(1); Y = np.eye(3)[y]
    return {f'{prefix}macro_F1': f1_score(y, pred, average='macro'),
            f'{prefix}recall_High': recall_score(y, pred, labels=[2], average='macro', zero_division=0),
            f'{prefix}presisi_High': (pred[y == 2] == 2).sum() / max((pred == 2).sum(), 1),
            f'{prefix}AUC_High': roc_auc_score(y == 2, P[:, 2]) if P[:, 2].std() > 0 else 0.5,
            f'{prefix}Brier': ((P - Y) ** 2).sum(1).mean(),
            f'{prefix}log_loss': log_loss(y, np.clip(P, 1e-6, 1), labels=[0, 1, 2])}

## D. Train → valid (setelan default; grid `C` untuk M1)
Protokol HPO resmi (§5.2): latih di `train` (600), nilai di `valid` (200). Di NB04 hanya `C` milik LR yang dipilih (4 nilai); model pohon memakai setelan default yang wajar untuk data kecil. HPO pohon ada di NB05.

Patokan NB03: LR set inti train → valid **0,698** (C = 0,5).

In [ ]:
Xtr, ytr = DEV[TR_MASK], y_dev[TR_MASK]; Xva, yva = DEV[VA_MASK], y_dev[VA_MASK]
grid = []
for c_ in [0.1, 0.3, 1, 3]:
    m = fit_model('M1', CORE, Xtr, ytr, C_=c_); grid.append({'C': c_, **metrics(yva, m.predict_proba(Xva))})
GRID = pd.DataFrame(grid); display(GRID.round(3))
LR_C = float(GRID.sort_values(['macro_F1', 'recall_High'], ascending=False).C.iloc[0])
print(f'C terpilih untuk M1: {LR_C}  (dipilih di valid → skor valid M1 sedikit optimistis; angka OOF di bagian E lebih jujur)')

t0 = time.time(); rows = []; VALID_P = {}
for code in MODELS:
    m = fit_model(code, CORE, Xtr, ytr); VALID_P[code] = m.predict_proba(Xva)
    rows.append({'kode': code, 'model': MODELS[code][0], **metrics(yva, VALID_P[code])})
VALID = pd.DataFrame(rows).set_index('kode')
VALID.to_csv(OUT / 'valid_results.csv')
display(VALID.round(3))
print(f'selesai dalam {time.time() - t0:.0f} dtk')

## E. Prediksi out-of-fold (OOF) — 5-fold `cv_fold` di train + valid (800)
- Setiap fold: fit pipeline lengkap (termasuk PeerZScaler), prediksi fold yang ditinggalkan.
- Hasil: 800 × 3 probabilitas per model. Ini dasar **semua** keputusan lapis 2–4 dan skor dashboard nasabah dev.
- Rentang 95% dari bootstrap (2.000 sampel ulang) pada prediksi OOF.
- Test & tanpa label: model **final** dilatih di seluruh 800 baris dev (test tidak pernah ikut), lalu memprediksi test (disimpan, **tidak dinilai**) dan tanpa label.

In [ ]:
FOLDS = [(np.flatnonzero(DEV.cv_fold.values != k), np.flatnonzero(DEV.cv_fold.values == k)) for k in range(5)]
OOF = {}; t0 = time.time()
for code in MODELS:
    P = np.zeros((len(DEV), 3))
    for tr, va in FOLDS:
        P[va] = fit_model(code, CORE, DEV.iloc[tr], y_dev[tr]).predict_proba(DEV.iloc[va])
    OOF[code] = P
print(f'OOF {len(MODELS)} model selesai dalam {time.time() - t0:.0f} dtk')

rng = np.random.default_rng(SEED); BOOT = rng.integers(0, len(DEV), size=(2000, len(DEV)))
def boot_ci(y, pred, fn):
    v = np.array([fn(y[b], pred[b]) for b in BOOT]); return np.percentile(v, [2.5, 97.5])
def _cm3(y, p): return np.bincount(3 * np.asarray(y) + np.asarray(p), minlength=9).reshape(3, 3)
def f1m(y, p):
    """macro-F1 3 kelas (= f1_score(average='macro', labels=[0,1,2], zero_division=0)), versi numpy cepat untuk bootstrap."""
    cm = _cm3(y, p); tp = np.diag(cm); den = cm.sum(0) + cm.sum(1)
    return np.mean(np.divide(2 * tp, den, out=np.zeros(3), where=den > 0))
def recH(y, p):
    cm = _cm3(y, p); return cm[2, 2] / cm[2].sum() if cm[2].sum() else 0.0
assert abs(f1m(y_dev, OOF['M1'].argmax(1)) - f1_score(y_dev, OOF['M1'].argmax(1), average='macro')) < 1e-12
rows = []
for code, P in OOF.items():
    pred = P.argmax(1); lo, hi = boot_ci(y_dev, pred, f1m); rlo, rhi = boot_ci(y_dev, pred, recH)
    fold_f1 = [f1m(y_dev[va], pred[va]) for _, va in FOLDS]
    rows.append({'kode': code, 'model': MODELS[code][0], **metrics(y_dev, P), 'F1_lo95': lo, 'F1_hi95': hi, 'recall_lo95': rlo, 'recall_hi95': rhi, 'sd_F1_antarfold': np.std(fold_f1)})
OOFRES = pd.DataFrame(rows).set_index('kode'); OOFRES.to_csv(OUT / 'oof_results.csv')
display(OOFRES[['model', 'macro_F1', 'F1_lo95', 'F1_hi95', 'sd_F1_antarfold', 'recall_High', 'presisi_High', 'AUC_High', 'Brier', 'log_loss']].round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.6), sharey=True)
order = [k for k in OOFRES.index if k != 'M0']; yy = np.arange(len(order))[::-1]
for a_, met, lo, hi, ttl in [(ax[0], 'macro_F1', 'F1_lo95', 'F1_hi95', 'E1a · macro-F1 OOF (rentang 95% bootstrap)'),
                             (ax[1], 'recall_High', 'recall_lo95', 'recall_hi95', 'E1b · recall High OOF (argmax)')]:
    d_ = OOFRES.loc[order]
    a_.errorbar(d_[met], yy, xerr=[d_[met] - d_[lo], d_[hi] - d_[met]], fmt='o', color=C['s1'], ms=7, capsize=2, lw=1.5)
    a_.axvline(d_.loc['M1', met], color=C['muted'], ls='--', lw=1)
    a_.set_title(ttl); a_.grid(axis='y', visible=False)
ax[0].set_yticks(yy); ax[0].set_yticklabels([f'{k} · {MODELS[k][0]}' for k in order])
ax[0].axvline(.83, color=C['s8'], ls=':', lw=1); ax[0].text(.83, yy.max() + .35, ' oracle ≈ 0,83', fontsize=7.5, color=C['s8'])
ax[0].set_xlim(.55, .87); ax[1].set_xlim(.5, 1.0)
ax[0].text(d_.loc['M1', 'macro_F1'], -0.9, ' champion M1', fontsize=7.5, color=C['muted'])
fig.tight_layout(); save_fig(fig, 'E1_oof_models'); plt.show()

### E2. Model final & prediksi test / tanpa label
Populasi dan set fitur:
- test (nasabah kredit, 200) → `set_core`;
- nasabah non-kredit (400) → `set_noncredit`;
- prospect (600) → `set_prospect` (tanpa *importance weighting*; itu di NB05).

Semua model (champion & challenger) menghasilkan prediksi supaya challenger bisa berjalan dalam **mode shadow**. Kinerja set turunan diukur sebagai informasi di OOF nasabah kredit (fitur yang hilang memang tidak dipakai).

In [ ]:
POP_SET = {'Existing credit client': 'set_core', 'Existing non-credit client': 'set_noncredit', 'Prospect': 'set_prospect'}
SCORE_POP['feature_set'] = SCORE_POP.customer_type.map(POP_SET)
FINAL = {}; POP_P = {code: np.zeros((len(SCORE_POP), 3)) for code in MODELS}; t0 = time.time()
for code in MODELS:
    for set_name, cols in FEATURE_SETS.items():
        m = fit_model(code, cols, DEV, y_dev); FINAL[(code, set_name)] = m
        ix = np.flatnonzero(SCORE_POP.feature_set.values == set_name)
        POP_P[code][ix] = m.predict_proba(SCORE_POP.iloc[ix])
print(f'model final {len(FINAL)} fit selesai dalam {time.time() - t0:.0f} dtk')

# OOF set turunan untuk semua model — dipakai (1) sebagai informasi kinerja, (2) untuk menghitung q̂ conformal & ambang P(High)
# per set fitur, karena model set_noncredit / set_prospect punya skala probabilitas sendiri
SET_OOF = {(k, 'set_core'): OOF[k] for k in MODELS}
rows = []
for code in [k for k in MODELS if k != 'M0']:
    for set_name in ['set_noncredit', 'set_prospect']:
        P = np.zeros((len(DEV), 3))
        for tr, va in FOLDS:
            P[va] = fit_model(code, FEATURE_SETS[set_name], DEV.iloc[tr], y_dev[tr]).predict_proba(DEV.iloc[va])
        SET_OOF[(code, set_name)] = P
        rows.append({'kode': code, 'set': set_name, 'n_fitur': len(FEATURE_SETS[set_name]), **metrics(y_dev, P)})
SUBSET = pd.concat([pd.DataFrame([{'kode': k, 'set': 'set_core', 'n_fitur': len(CORE), **metrics(y_dev, OOF[k])} for k in MODELS if k != 'M0']), pd.DataFrame(rows)])
SUBSET.to_csv(OUT / 'oof_feature_set_variants.csv', index=False)
display(SUBSET.pivot(index='set', columns='kode', values='macro_F1').loc[list(FEATURE_SETS)].round(3).assign(n_fitur=[len(v) for v in FEATURE_SETS.values()]))
print(f'OOF set turunan selesai; total {time.time() - t0:.0f} dtk sejak model final')
print('Catatan: set_prospect diukur pada nasabah kredit (fitur transaksi sengaja dibuang). Pergeseran distribusi prospect belum diperhitungkan → NB05/NB06.')

# simpan semua prediksi (OOF, test, tanpa label) per model
long = []
for code in MODELS:
    long.append(pd.DataFrame({'company_id': DEV.company_id, 'split': DEV.split, 'customer_type': DEV.customer_type, 'pred_source': 'oof',
                              'feature_set': 'set_core', 'model': code, 'p_low': OOF[code][:, 0], 'p_medium': OOF[code][:, 1], 'p_high': OOF[code][:, 2]}))
    long.append(pd.DataFrame({'company_id': SCORE_POP.company_id, 'split': SCORE_POP.split, 'customer_type': SCORE_POP.customer_type,
                              'pred_source': np.where(SCORE_POP.split == 'test', 'test', 'unlabeled'), 'feature_set': SCORE_POP.feature_set, 'model': code,
                              'p_low': POP_P[code][:, 0], 'p_medium': POP_P[code][:, 1], 'p_high': POP_P[code][:, 2]}))
ALLPRED = pd.concat(long, ignore_index=True)
print('prediksi tersimpan:', write_table(ALLPRED, PRED / 'risk_predictions_all_models'), '|', ALLPRED.shape)

## F. Uji logika bisnis (kriteria 4 gerbang)
Untuk setiap fitur yang punya arah di §5.3: naikkan nilainya **1 SD** (fitur biner: ubah 0 → 1) pada baris dev yang tidak kosong, lalu lihat perubahan rata-rata P(High) pada model final `set_core`.
- **Lolos** bila rata-rata ΔP(High) searah dengan logika bisnis, atau praktis nol (|Δ| < 0,002).
- Kolom `*_pz` tidak diuji terpisah: nilainya dihitung ulang dari rasio mentah di dalam pipeline, jadi menaikkan `fin_log_der` sekaligus menaikkan `der_pz`.
- Model pohon ordinal diberi batasan monoton, jadi diharapkan lolos 100%. Uji ini paling berguna untuk LR dan model tanpa batasan.

In [ ]:
TEST_FEATS = [c for c in CORE if MONO.get(c, 0) != 0 and c not in PZ_OUT]
rows = []
for code in [k for k in MODELS if k != 'M0']:
    m = FINAL[(code, 'set_core')]
    for f in TEST_FEATS:
        ok = DEV[f].notna().values; X0 = DEV[ok]; X1 = X0.copy()
        binary = set(DEV[f].dropna().unique()) <= {0, 1}
        if binary:
            X0 = X0[X0[f] == 0]; X1 = X0.copy(); X1[f] = 1
        else:
            X1[f] = X1[f] + DEV[f].std()
        if len(X0) == 0: continue
        d = m.predict_proba(X1)[:, 2] - m.predict_proba(X0)[:, 2]; s = MONO[f]
        rows.append({'kode': code, 'fitur': f, 'arah': '+' if s > 0 else '−', 'mean_dP_High': d.mean(), 'pct_baris_salah_arah': 100 * (s * d < -1e-4).mean(),
                     'lolos': bool(s * d.mean() >= 0 or abs(d.mean()) < 0.002)})
LOGIC = pd.DataFrame(rows); LOGIC.to_csv(OUT / 'logic_test.csv', index=False)
LOGIC_SUM = LOGIC.groupby('kode').agg(n_uji=('lolos', 'size'), n_lolos=('lolos', 'sum'), pct_baris_salah_arah=('pct_baris_salah_arah', 'mean'))
LOGIC_SUM['lolos_semua'] = LOGIC_SUM.n_uji == LOGIC_SUM.n_lolos
display(LOGIC_SUM.round(1))
fails = LOGIC[~LOGIC.lolos]
if len(fails): print('fitur yang gagal uji arah:'); display(fails.round(4))

## G. Lapis 2–3 versi 0 (kalibrasi & keputusan disempurnakan di NB05)
### G1. Kurva risk–coverage (selective classification; Geifman & El-Yaniv, 2017)
Kasus diurutkan dari yang paling yakin (probabilitas kelas tertinggi). Pada setiap tingkat cakupan (% kasus yang diputuskan otomatis), hitung macro-F1 kasus yang diputuskan otomatis. Kurva ini menunjukkan berapa banyak kasus yang bisa diputuskan mesin pada kualitas tertentu.

In [ ]:
COV = np.linspace(.2, 1, 17)
def risk_coverage(y, P):
    conf = P.max(1); order_ = np.argsort(-conf); out = []
    for c in COV:
        k = int(round(c * len(y))); ix = order_[:k]
        out.append(f1_score(y[ix], P[ix].argmax(1), average='macro', labels=[0, 1, 2], zero_division=0))
    return np.array(out)
RC = pd.DataFrame({code: risk_coverage(y_dev, OOF[code]) for code in MODELS if code != 'M0'}, index=np.round(COV, 2)); RC.index.name = 'cakupan'
RC.to_csv(OUT / 'risk_coverage_oof.csv')
display(RC.loc[[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]].round(3))

### G2. Ambang P(High) (lapis 3) dan matriks biaya
Ambang P(High) dipilih di OOF supaya **recall High ≥ target** (90%, gaya Neyman–Pearson; usulan rencana 85–90%). Kelas lain diputuskan dengan argmax Low vs Medium. Biaya dari matriks §5.4 (baris = sebenarnya, kolom = prediksi):

| sebenarnya \ prediksi | Low | Medium | High |
|---|---|---|---|
| Low | 0 | 1 | 2 |
| Medium | 3 | 0 | 1 |
| High | 10 | 5 | 0 |

In [ ]:
COST = np.array([[0, 1, 2], [3, 0, 1], [10, 5, 0]])
TARGET_RECALL = 0.90
def high_threshold(y, P, target):
    """ambang P(High) terbesar yang masih memberi recall High ≥ target."""
    ph = P[y == 2, 2]; return np.sort(ph)[::-1][int(np.ceil(target * len(ph))) - 1]
def decide(P, t_high):
    pred = np.where(P[:, 1] > P[:, 0], 1, 0); pred[P[:, 2] >= t_high] = 2; return pred
rows = []
for code in [k for k in MODELS if k != 'M0']:
    P = OOF[code]
    for rule, pred, t in [('argmax', P.argmax(1), np.nan)] + [(f'ambang recall ≥ {int(100 * tr)}%', decide(P, high_threshold(y_dev, P, tr)), high_threshold(y_dev, P, tr)) for tr in (0.85, 0.90)]:
        cm = confusion_matrix(y_dev, pred, labels=[0, 1, 2])
        rows.append({'kode': code, 'aturan': rule, 'ambang_P_High': t, 'macro_F1': f1m(y_dev, pred), 'recall_High': recH(y_dev, pred),
                     'flag_High': int((pred == 2).sum()), 'biaya_total': int((cm * COST).sum()), 'biaya_per_nasabah': (cm * COST).sum() / len(y_dev)})
L3 = pd.DataFrame(rows); L3.to_csv(OUT / 'decision_rules_oof.csv', index=False)
display(L3.round(3))

### G3. Conformal prediction v0 (LAC, α = 10%; Angelopoulos & Bates, 2021)
Skor ketidaksesuaian s = 1 − P(kelas sebenarnya) di OOF champion. Ambang q̂ = kuantil ke-⌈(n+1)(1−α)⌉/n. Himpunan kelas = {k : P(k) ≥ 1 − q̂}.
- Himpunan 1 kelas dan sama dengan keputusan lapis 3 → **Otomatis**;
- selain itu (himpunan > 1 kelas, kosong, atau bertentangan dengan ambang High) → **Review analis**.

Versi 0: q̂ dihitung dan dinilai di OOF yang sama (cakupan sedikit optimistis), tanpa kalibrasi. NB05 memakai kalibrasi + conformal terpisah, dan *weighted conformal* untuk prospect.

In [ ]:
ALPHA = 0.10
def conformal_q(y, P, alpha=ALPHA):
    s = 1 - P[np.arange(len(y)), y]; n = len(s); return np.quantile(s, min(1, np.ceil((n + 1) * (1 - alpha)) / n), method='higher')
def conformal_sets(P, q): return P >= 1 - q
def decision_table(y, P, q, t_high):
    S = conformal_sets(P, q); pred = decide(P, t_high); size = S.sum(1)
    auto = (size == 1) & S[np.arange(len(P)), pred]
    return S, pred, auto
rows = []
for code in [k for k in MODELS if k != 'M0']:
    P = OOF[code]; q = conformal_q(y_dev, P); t = high_threshold(y_dev, P, TARGET_RECALL); S, pred, auto = decision_table(y_dev, P, q, t)
    rows.append({'kode': code, 'q_hat': q, 'cakupan': S[np.arange(len(y_dev)), y_dev].mean(), 'rata2_ukuran_himpunan': S.sum(1).mean(),
                 'pct_otomatis': 100 * auto.mean(), 'macro_F1_otomatis': f1_score(y_dev[auto], pred[auto], average='macro', labels=[0, 1, 2], zero_division=0),
                 'akurasi_otomatis': (pred[auto] == y_dev[auto]).mean(), 'High_lolos_otomatis': int(((y_dev == 2) & auto & (pred != 2)).sum()),
                 'macro_F1_semua_ambang': f1m(y_dev, pred), 'recall_High_ambang': recH(y_dev, pred)})
CONF = pd.DataFrame(rows).set_index('kode'); CONF.to_csv(OUT / 'conformal_v0_oof.csv')
display(CONF.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
for code, col in zip(['M1', 'M2', 'M3', 'M4', 'M3x'], [C['s1'], C['s3'], C['s2'], C['s7'], C['muted']]):
    ax[0].plot(100 * RC.index, RC[code], color=col, lw=2 if code == 'M1' else 1.5, ls='--' if code == 'M3x' else '-', label=f'{code} · {MODELS[code][0].split(" (")[0]}')
    r = CONF.loc[code]; ax[0].plot(r.pct_otomatis, r.macro_F1_otomatis, 'o', color=col, ms=8, mec=C['surface'], mew=2)
ax[0].set_xlabel('% kasus diputuskan otomatis (cakupan)'); ax[0].set_ylabel('macro-F1 kasus otomatis'); ax[0].set_title('G1 · Kurva risk–coverage (OOF)')
ax[0].plot([], [], 'o', color=C['ink2'], ms=7, label='● titik operasi conformal α = 10%')
ax[0].legend(loc='upper center', bbox_to_anchor=(.5, -.2), ncol=3, fontsize=7.5)
d_ = L3[L3.aturan != 'argmax']
for code, col in zip(['M1', 'M2', 'M3', 'M4'], [C['s1'], C['s3'], C['s2'], C['s7']]):
    e = d_[d_.kode == code]; ax[1].plot(e.flag_High, e.macro_F1, '-o', color=col, ms=6, lw=1.5, label=code)
    for _, r in e.iterrows(): ax[1].annotate(r.aturan.split('≥ ')[1], (r.flag_High, r.macro_F1), fontsize=7, color=C['ink2'], xytext=(4, 3), textcoords='offset points')
ax[1].set_xlabel('jumlah flag High (dari 800)'); ax[1].set_ylabel('macro-F1'); ax[1].set_title('G2 · Harga mengejar recall High 85% → 90%'); ax[1].legend(ncol=4)
fig.tight_layout(); save_fig(fig, 'G1_risk_coverage_threshold'); plt.show()

### G4. Cek per segmen (§5.5b): satu model global, dilaporkan per segmen
Model risiko **tidak** dipecah per segmen (label hanya 800, kasus High per segmen terlalu sedikit). Kinerja OOF dilaporkan per segmen ukuran.
- Segmen hasil clustering baru dibuat di **NB08**. Sementara itu dipakai `size_class_reference` (86% sama dengan segmen clustering, §8b.4) **hanya untuk pelaporan**, bukan fitur.
- Cakupan conformal per segmen menunjukkan apakah *Mondrian conformal* per segmen (NB05) diperlukan.

In [ ]:
SEG = DEV.size_class_reference.values
rows = []
for code in ['M1', 'M2', 'M3', 'M4']:
    P = OOF[code]; q = conformal_q(y_dev, P); S = conformal_sets(P, q)
    for s in pd.unique(SEG):
        ix = SEG == s
        rows.append({'kode': code, 'segmen (sementara)': s, 'n': int(ix.sum()), 'n_High': int((y_dev[ix] == 2).sum()),
                     'macro_F1': f1_score(y_dev[ix], P[ix].argmax(1), average='macro', labels=[0, 1, 2], zero_division=0),
                     'recall_High': recH(y_dev[ix], P[ix].argmax(1)) if (y_dev[ix] == 2).any() else np.nan,
                     'cakupan_conformal': S[ix][np.arange(ix.sum()), y_dev[ix]].mean()})
SEGRES = pd.DataFrame(rows); SEGRES.to_csv(OUT / 'oof_by_segment.csv', index=False)
piv = SEGRES.pivot(index='segmen (sementara)', columns='kode', values=['macro_F1', 'recall_High', 'cakupan_conformal']).round(3)
piv.columns = [f'{a} {b}' for a, b in piv.columns]
display(SEGRES.drop_duplicates('segmen (sementara)').set_index('segmen (sementara)')[['n', 'n_High']].join(piv))
print('Cakupan conformal global ±90%; bila satu segmen jauh di bawah 90%, NB05 memakai Mondrian conformal per segmen.')

## H. Gerbang promosi v0 — champion vs challenger (lapis 4)
Champion awal = **M1** (paling mudah dijelaskan). Setiap challenger dibandingkan di **OOF** dengan 4 kriteria (§7.4):

| # | Kriteria | Cara uji di NB04 |
|---|---|---|
| 1 | macro-F1 lebih tinggi **dan signifikan** | bootstrap berpasangan 2.000× pada prediksi OOF (argmax); p = P(selisih ≤ 0) < 0,05 |
| 2 | recall High tidak turun | recall argmax ≥ champion − 0,02 (target 90% dicapai lewat ambang lapis 3, bagian G2) |
| 3 | kalibrasi tidak memburuk | Brier ≤ champion (probabilitas mentah; kalibrasi di NB05) |
| 4 | lolos uji logika bisnis | 100% fitur lolos uji arah (bagian F) |

Informasi tambahan (bukan syarat): % otomatis pada titik operasi conformal yang sama.

In [ ]:
CHAMP = 'M1'
def gate(champ, chal):
    pc, ph = OOF[champ].argmax(1), OOF[chal].argmax(1)
    diff = np.array([f1m(y_dev[b], ph[b]) - f1m(y_dev[b], pc[b]) for b in BOOT]); p = (diff <= 0).mean()
    rc, rh = recH(y_dev, pc), recH(y_dev, ph); bc, bh = OOFRES.loc[champ, 'Brier'], OOFRES.loc[chal, 'Brier']
    k1 = bool(diff.mean() > 0 and p < 0.05); k2 = bool(rh >= rc - 0.02); k3 = bool(bh <= bc); k4 = bool(LOGIC_SUM.loc[chal, 'lolos_semua'])
    return {'challenger': chal, 'ΔF1': diff.mean(), 'ΔF1_lo95': np.percentile(diff, 2.5), 'ΔF1_hi95': np.percentile(diff, 97.5), 'p_bootstrap': p,
            'K1_F1_signifikan': k1, 'recall_High': rh, 'K2_recall': k2, 'Brier': bh, 'K3_Brier': k3, 'K4_logika': k4,
            'pct_otomatis': CONF.loc[chal, 'pct_otomatis'], 'lolos_gerbang': k1 and k2 and k3 and k4}
GATE = pd.DataFrame([gate(CHAMP, c) for c in ['M2', 'M3', 'M4', 'M3x']]).set_index('challenger')
GATE.to_csv(OUT / 'promotion_gate_v0.csv')
card = GATE[['ΔF1', 'p_bootstrap', 'K1_F1_signifikan', 'K2_recall', 'K3_Brier', 'K4_logika', 'lolos_gerbang']].copy()
for c in ['K1_F1_signifikan', 'K2_recall', 'K3_Brier', 'K4_logika', 'lolos_gerbang']: card[c] = card[c].map({True: '✅', False: '❌'})
print(f'Champion: {CHAMP} · {MODELS[CHAMP][0]} | macro-F1 OOF {OOFRES.loc[CHAMP, "macro_F1"]:.3f}, recall High {OOFRES.loc[CHAMP, "recall_High"]:.3f}, Brier {OOFRES.loc[CHAMP, "Brier"]:.3f}')
display(card.round(3))
promoted = GATE[GATE.lolos_gerbang]
if len(promoted):
    CHAMP = promoted.sort_values('ΔF1', ascending=False).index[0]; print(f'→ challenger {CHAMP} lolos semua kriteria dan menjadi champion baru')
else:
    print(f'→ tidak ada challenger yang lolos semua kriteria; {CHAMP} tetap champion. Challenger berjalan dalam mode shadow.')

# registri model
reg = []
for code, (name, fam, sub) in MODELS.items():
    status = ('champion' if code == CHAMP else 'lantai' if code == 'M0' else 'pembanding (tanpa batasan monoton)' if code == 'M3x' else 'challenger (shadow)')
    g = GATE.loc[code] if code in GATE.index else None
    reg.append({'kode': code, 'model': name, 'keluarga': fam, 'implementasi': sub or name, 'feature_set': 'set_core (+ set_noncredit / set_prospect untuk tanpa label)',
                'hyperparameter': f'C={LR_C}' if code == 'M1' else 'default NB04',
                'valid_macro_F1': VALID.loc[code, 'macro_F1'], 'valid_recall_High': VALID.loc[code, 'recall_High'],
                'oof_macro_F1': OOFRES.loc[code, 'macro_F1'], 'oof_recall_High': OOFRES.loc[code, 'recall_High'], 'oof_Brier': OOFRES.loc[code, 'Brier'],
                'oof_AUC_High': OOFRES.loc[code, 'AUC_High'], 'logika_lolos': None if code == 'M0' else bool(LOGIC_SUM.loc[code, 'lolos_semua']),
                'gerbang_vs_champion': None if g is None else ('lolos' if g.lolos_gerbang else 'tidak lolos'), 'status': status, 'versi': 'v0 (NB04)'})
REGISTRY = pd.DataFrame(reg); REGISTRY.to_csv(OUT / 'model_registry.csv', index=False)
display(REGISTRY[['kode', 'implementasi', 'oof_macro_F1', 'oof_recall_High', 'oof_Brier', 'logika_lolos', 'gerbang_vs_champion', 'status']].round(3))

## I. Skor risiko v0 untuk dashboard — `outputs/scores/risk_scores.csv`
- Nasabah dev (800) memakai prediksi **OOF** champion (bukan prediksi in-sample).
- Test (200) dan tanpa label (1.000) memakai model final champion (dilatih di 800 dev; test tidak pernah ikut).
- `conformal_set` & `decision` memakai q̂ dan ambang P(High) dari OOF champion **per set fitur** (model `set_prospect` punya skala probabilitas sendiri, jadi ambangnya dihitung dari OOF set itu). Pergeseran distribusi prospect belum dikoreksi: *importance weighting* + *weighted conformal* di NB05.
- **3 alasan utama**: atribusi oklusi — ubah satu fitur ke median dev, ukur turunnya probabilitas kelas terpilih. Ini pengganti sementara SHAP (dipasang di NB05), bekerja untuk model apa pun. Rasio mentah dan versi relatif-peernya dihitung sebagai satu alasan.

In [ ]:
LABEL = {'size_ord': 'ukuran usaha', 'fin_log_der': 'DER (utang/ekuitas)', 'fin_log_cr': 'current ratio', 'fin_npm': 'margin laba bersih',
         'fin_sicr': 'interest coverage', 'has_report': 'laporan keuangan diserahkan', 'is_audited': 'laporan diaudit',
         'pay_late_6m': 'gaji telat (≥ tgl 28), 6 bln', 'pay_late_12m': 'gaji telat (≥ tgl 28), 12 bln', 'pay_late_30m': 'gaji telat (≥ tgl 28), 30 bln',
         'tax_late_6m': 'pajak telat (> tgl 15), 6 bln', 'tax_late_12m': 'pajak telat (> tgl 15), 12 bln', 'tax_late_30m': 'pajak telat (> tgl 15), 30 bln',
         'pay_split_12m': 'gaji dicicil, 12 bln', 'timing_delta': 'keterlambatan bayar memburuk (6 vs 12 bln)', 'out_in_6m': 'outflow ÷ inflow 6 bln',
         'out_in_12m': 'outflow ÷ inflow 12 bln', 'inflow_cv_sa': 'volatilitas inflow', 'inflow_growth_sa': 'pertumbuhan inflow',
         'max_inflow_z_12m': 'lonjakan inflow terbesar 12 bln', 'capture_ratio': 'porsi omzet yang lewat BNI', 'divergence': 'inflow BNI vs omzet (divergensi)',
         'utilization': 'utilisasi fasilitas kredit', 'log_limit': 'besar limit kredit', 'top_buyer_share': 'porsi pembeli terbesar',
         'exp_shocked_anchor': 'paparan ke anchor yang melemah'}
PCT = {'pay_late_6m', 'pay_late_12m', 'pay_late_30m', 'tax_late_6m', 'tax_late_12m', 'tax_late_30m', 'pay_split_12m', 'utilization', 'top_buyer_share', 'capture_ratio', 'exp_shocked_anchor'}
MED = DEV[CORE].median()

def occlusion_reasons(model, X, cols, pred, top=3):
    """Kontribusi fitur j = P_pred(x) − P_pred(x dengan fitur j = median dev), keduanya dari model yang sama.
    Rasio mentah (mis. fin_log_der) ikut menggeser versi relatif-peernya karena pz dihitung ulang di pipeline.
    Hanya kontribusi ≥ 0,01 yang ditampilkan."""
    feats = [c for c in cols if c not in PZ_OUT]; contrib = np.zeros((len(X), len(feats))); rows_ = np.arange(len(X))
    P = model.predict_proba(X)
    for j, f in enumerate(feats):
        X1 = X.copy(); X1[f] = MED[f]
        contrib[:, j] = P[rows_, pred] - model.predict_proba(X1)[rows_, pred]
    contrib[X[feats].isna().values & ~np.isin(feats, ['has_report', 'is_audited'])] = 0     # nilai kosong bukan alasan (kecuali indikator laporan)
    out = []
    for i in rows_:
        order_ = np.argsort(-contrib[i])[:top]; rs = []
        for j in order_:
            if contrib[i, j] < 0.01: break
            f = feats[j]; v = X[f].iloc[i]
            if f.startswith('ind_'):
                txt = f'industri {f[4:].replace("_", " ")}'
            elif f == 'has_report':
                txt = 'laporan keuangan FY2025 tidak diserahkan' if v == 0 else 'laporan keuangan diserahkan'
            elif f == 'is_audited':
                txt = 'laporan tidak diaudit' if v == 0 else 'laporan diaudit'
            else:
                fmt = (lambda x: f'{100 * x:.0f}%') if f in PCT else (lambda x: f'{x:.2f}')
                txt = f'{LABEL.get(f, f)} {"tinggi" if v > MED[f] else "rendah"} ({fmt(v)}; median {fmt(MED[f])})'
            rs.append(f'{txt} → +{100 * contrib[i, j]:.0f} poin P({RISK_ORDER[pred[i]]})')
        out.append(rs + [''] * (top - len(rs)))
    return out

P_ch = OOF[CHAMP]
def set_rules(code):
    """q̂ conformal & ambang P(High) per set fitur, dari OOF model yang memakai set itu (skala probabilitasnya berbeda)."""
    return {sn: (conformal_q(y_dev, SET_OOF[(code, sn)]), high_threshold(y_dev, SET_OOF[(code, sn)], TARGET_RECALL)) for sn in FEATURE_SETS}
QT = set_rules(CHAMP); q_ch, t_ch = QT['set_core']
display(pd.DataFrame(QT, index=['q_hat', 'ambang_P_High']).T.round(3))
parts = []
# dev: prediksi OOF; alasan dari model fold yang sama tidak praktis → pakai model final set_core (dicatat di kolom reason_source)
for pop_name, X, P, src, set_name in [('dev', DEV, P_ch, 'oof', 'set_core')] + \
        [(s, SCORE_POP[SCORE_POP.feature_set == s], POP_P[CHAMP][SCORE_POP.feature_set.values == s], None, s) for s in FEATURE_SETS]:
    if len(X) == 0: continue
    X = X.reset_index(drop=True); S, pred, auto = decision_table(np.zeros(len(X), int), P, *QT[set_name])
    reasons = occlusion_reasons(FINAL[(CHAMP, set_name)], X, FEATURE_SETS[set_name], pred)
    parts.append(pd.DataFrame({'company_id': X.company_id, 'split': X.split, 'customer_type': X.customer_type,
                               'pred_source': src or np.where(X.split == 'test', 'test', 'unlabeled'),
                               'model_version': f'{CHAMP}-v0-NB04', 'feature_set': set_name,
                               'p_low': P[:, 0].round(4), 'p_medium': P[:, 1].round(4), 'p_high': P[:, 2].round(4),
                               'pred_class': [RISK_ORDER[k] for k in pred],
                               'conformal_set': ['{' + ','.join(RISK_ORDER[k] for k in range(3) if s[k]) + '}' for s in S],
                               'decision': np.where(auto, 'Otomatis', 'Review analis'),
                               'reason_1': [r[0] for r in reasons], 'reason_2': [r[1] for r in reasons], 'reason_3': [r[2] for r in reasons],
                               'reason_source': 'oklusi, model final ' + set_name}))
RISK_SCORES = pd.concat(parts, ignore_index=True).sort_values('company_id').reset_index(drop=True)
assert RISK_SCORES.company_id.is_unique and len(RISK_SCORES) == 2000
RISK_SCORES.to_csv(SCORES / 'risk_scores.csv', index=False)
print(f'risk_scores.csv: {RISK_SCORES.shape} | champion {CHAMP} | set_core: q̂ = {q_ch:.3f}, ambang P(High) = {t_ch:.3f}')
display(pd.crosstab([RISK_SCORES.customer_type, RISK_SCORES.pred_source], RISK_SCORES.decision, margins=True))
dist = pd.crosstab(RISK_SCORES.customer_type, RISK_SCORES.pred_class)[RISK_ORDER]
display(dist.join((100 * dist.High / dist.sum(1)).round(1).rename('% High')))
print('Catatan: ambang dipilih untuk recall High 90%, jadi ±25–30% nasabah kredit di-flag High (label sebenarnya ±14%). '
      'Persentase prospect yang lebih tinggi sebagian karena pergeseran distribusi (prospect ≠ nasabah) — dikoreksi & dinilai ke GT di NB05–06.')
display(RISK_SCORES[RISK_SCORES.pred_class == 'High'].head(4)[['company_id', 'customer_type', 'p_high', 'conformal_set', 'decision', 'reason_1', 'reason_2', 'reason_3']])

### I2. Mode shadow (§7.5) — `shadow_log`
Setiap challenger dibungkus lapis 2–3 yang sama (q̂ dan ambang P(High) dari OOF-nya sendiri, per set fitur). Log mencatat berapa keputusan yang akan berbeda bila challenger menggantikan champion. Tabel ini bahan halaman audit di dashboard.

In [ ]:
champ_cls = RISK_SCORES.set_index('company_id').pred_class; champ_dec = RISK_SCORES.set_index('company_id').decision
rows = []; shadow_rows = []
for code in [k for k in MODELS if k not in ('M0', CHAMP)]:
    qt = set_rules(code)
    sub = ALLPRED[ALLPRED.model == code].set_index('company_id').loc[champ_cls.index]
    P = sub[['p_low', 'p_medium', 'p_high']].values; pred = np.zeros(len(P), int); auto = np.zeros(len(P), bool)
    for sn in FEATURE_SETS:
        ix = (sub.feature_set == sn).values
        if ix.any(): _, pred[ix], auto[ix] = decision_table(np.zeros(ix.sum(), int), P[ix], *qt[sn])
    cls = pd.Series([RISK_ORDER[k] for k in pred], index=sub.index); dec = pd.Series(np.where(auto, 'Otomatis', 'Review analis'), index=sub.index)
    diff = cls != champ_cls
    for pop, m_ in [('semua', slice(None)), ('dev (OOF)', sub.pred_source == 'oof'), ('test', sub.pred_source == 'test'), ('tanpa label', sub.pred_source == 'unlabeled')]:
        mm = np.ones(len(sub), bool) if isinstance(m_, slice) else m_.values
        rows.append({'challenger': code, 'populasi': pop, 'n': int(mm.sum()), 'kelas_berbeda': int(diff[mm].sum()),
                     'jadi_High_baru': int(((cls == 'High') & (champ_cls != 'High'))[mm].sum()), 'High_hilang': int(((cls != 'High') & (champ_cls == 'High'))[mm].sum()),
                     'keputusan_berbeda': int((dec != champ_dec)[mm].sum())})
    d_ = sub[diff.values].assign(champion_class=champ_cls[diff.values], challenger_class=cls[diff.values])
    shadow_rows.append(d_.reset_index()[['company_id', 'split', 'pred_source', 'champion_class', 'challenger_class']].assign(challenger=code, champion=CHAMP))
SHADOW = pd.DataFrame(rows); SHADOW.to_csv(OUT / 'shadow_summary.csv', index=False)
pd.concat(shadow_rows).to_csv(SCORES / 'shadow_log.csv', index=False)
display(SHADOW[SHADOW.populasi == 'semua'].set_index('challenger').drop(columns='populasi'))
print('detail per perusahaan: outputs/scores/shadow_log.csv')

### I3. Simpan model final — `models/04_modelling/`
Semua model final (6 model × 3 set fitur = 18 pipeline) disimpan dengan `joblib`. Model ini dilatih di 800 nasabah dev (train + valid); test tidak pernah ikut. NB05–07 bisa langsung memakainya tanpa melatih ulang.
- `<kode>__<set>.joblib` — pipeline lengkap (PeerZScaler + estimator); input = DataFrame dari `load_features(PROJECT_ROOT, 'risk')`, output = `predict_proba` [Low, Medium, High].
- `manifest.json` — champion, versi model, `C` LR, set fitur, q̂ conformal & ambang P(High) per model × set, versi pustaka.
- `model_files.csv` — daftar berkas + status di registri.

Cara memakai di notebook lain:
```python
sys.path.insert(0, str(PROJECT_ROOT / 'src'))      # wajib: OrdinalMonotone ada di src/modelling.py
model = joblib.load(PROJECT_ROOT / 'models' / '04_modelling' / 'M1__set_core.joblib')
P = model.predict_proba(df)                        # kolom fitur dipilih sendiri oleh pipeline
```
Model joblib hanya aman dibaca dengan versi scikit-learn / XGBoost / LightGBM yang sama (tercatat di manifest).

In [ ]:
rows = []
for (code, set_name), m in FINAL.items():
    f = MODEL_DIR / f'{code}__{set_name}.joblib'
    joblib.dump(m, f, compress=3)
    rows.append({'kode': code, 'model': MODELS[code][0], 'feature_set': set_name, 'n_fitur': len(FEATURE_SETS[set_name]),
                 'status': REGISTRY.set_index('kode').loc[code, 'status'], 'file': f.relative_to(PROJECT_ROOT).as_posix(), 'ukuran_MB': round(f.stat().st_size / 1e6, 2)})
MODEL_FILES = pd.DataFrame(rows); MODEL_FILES.to_csv(MODEL_DIR / 'model_files.csv', index=False)

MANIFEST = {
    'champion': CHAMP, 'model_version': f'{CHAMP}-v0-NB04', 'dilatih_pada': 'train + valid (800 nasabah kredit); test tidak ikut',
    'risk_order': RISK_ORDER, 'LR_C': LR_C, 'target_recall_High': TARGET_RECALL, 'alpha_conformal': ALPHA,
    'populasi_ke_set': POP_SET, 'feature_sets': FEATURE_SETS,
    'aturan_keputusan': {code: {sn: {'q_hat': float(q), 'ambang_P_High': float(t)} for sn, (q, t) in set_rules(code).items()}
                         for code in MODELS if code != 'M0'},
    'versi_pustaka': {'python': sys.version.split()[0], 'scikit-learn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__,
                      'xgboost': xgb.__version__ if HAS_XGB else None, 'lightgbm': lgb.__version__ if HAS_LGB else None},
}
(MODEL_DIR / 'manifest.json').write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=1), encoding='utf-8')

# cek: model yang di-load ulang memberi prediksi yang sama
m_ = joblib.load(MODEL_DIR / f'{CHAMP}__set_core.joblib')
assert np.allclose(m_.predict_proba(DEV.head(50)), FINAL[(CHAMP, 'set_core')].predict_proba(DEV.head(50)))
print(f'{len(MODEL_FILES)} model tersimpan di {MODEL_DIR.relative_to(PROJECT_ROOT)} ({MODEL_FILES.ukuran_MB.sum():.1f} MB) | champion {CHAMP} ✓ load ulang')
display(MODEL_FILES.pivot(index='kode', columns='feature_set', values='ukuran_MB').join(MODEL_FILES.drop_duplicates('kode').set_index('kode')[['model', 'status']]))

## J. Deteksi anomali "Why Now"
### J1. Protokol (§8.1)
- Panel 1.400 nasabah × bulan, dipakai mulai **2024-07** (6 bulan pertama = pemanasan statistik online).
- Tidak ada label untuk melatih. Ground truth (`eval_targets/anomaly_events_monthly.csv`) dipakai **hanya untuk perusahaan non-test** (setara "valid") untuk memilih champion vs challenger. Perusahaan test baru dinilai di NB06.
- Ambang aturan **tidak di-tuning ke GT**: angka tetap dari rencana §8.2 (berasal dari EDA) atau persentil bulanan.

**Definisi evaluasi**
- **Alarm baru** = bulan pertama sebuah alarm menyala (bulan sebelumnya tidak menyala untuk jenis yang sama). Inilah yang dikirim ke RM.
- **Alarm benar** = perusahaan punya kejadian GT **jenis yang sama** yang aktif di bulan *t* ± 1. Untuk distress, bulan peringatan dini GT (`early_warning`, 3 bulan sebelum distress) dihitung benar.
- **Kejadian terdeteksi** (`recall_kejadian`) = ada alarm jenis yang sama sejak onset − 1 (distress: onset − 3) sampai sebulan setelah kejadian berakhir. **Recall dini** = alarm muncul paling lambat onset + 2.
- **Lead time distress** = onset − bulan alarm pertama di jendela [onset − 6, onset + 2].
- **Presisi top-k** = alarm baru diurutkan per bulan dengan skor prioritas (persentil sinyal di bulan itu), ambil k = 30 (kapasitas RM untuk ±1.200 nasabah non-test).

In [ ]:
PANEL = load_features(PROJECT_ROOT, 'anomaly')
PANEL = PANEL[PANEL.ym >= '2024-07'].reset_index(drop=True)
assert not {'relationship_end', 'data_visibility'} & set(PANEL.columns)
MONTHS = sorted(PANEL.ym.unique()); M_IX = {m: i for i, m in enumerate(MONTHS)}; PANEL['m'] = PANEL.ym.map(M_IX)
NONTEST = set(PANEL.company_id[PANEL.split != 'test']); N_NONTEST = len(NONTEST)
K_TOP = 30

EV = pd.read_csv(EVAL / 'anomaly_events_monthly.csv')
EV = EV[EV.company_id.isin(NONTEST)].copy()                      # perusahaan test TIDAK dibaca di NB04
EV['type'] = EV.event_type.replace({'early_warning': 'distress'})
ALL_M = {m: i for i, m in enumerate(sorted(set(EV.ym) | set(MONTHS)))}           # indeks bulan termasuk sebelum 2024-07
off = ALL_M[MONTHS[0]]; EV['m'] = EV.ym.map(ALL_M) - off
ACTIVE = set(zip(EV.company_id, EV.type, EV.m)); ACTIVE_ANY = set(zip(EV.company_id, EV.m))
ONSETS = EV[(EV.is_onset == 1) & (EV.event_type != 'early_warning')][['company_id', 'type', 'm']]
EV_END = EV.groupby(['company_id', 'type']).m.max().to_dict()                    # bulan terakhir kejadian aktif
assert not ONSETS.duplicated(['company_id', 'type']).any(), 'asumsi: satu kejadian per perusahaan × jenis'
print(f'panel: {len(PANEL):,} nasabah-bulan ({PANEL.company_id.nunique()} nasabah × {len(MONTHS)} bulan) | non-test {N_NONTEST}')
display(ONSETS.groupby('type').size().rename('kejadian GT (non-test)').to_frame().T)

### J2. Champion: aturan transparan per jenis kejadian (§8.2)
| Jenis | Aturan (data ≤ bulan *t*) | Skor prioritas |
|---|---|---|
| Lonjakan sesaat | `spike_candidate == 1` | persentil `inflow_resid_z` |
| Pembeli utama keluar | `buyer_exit_share ≥ 0,15` | persentil `buyer_exit_share` |
| Efek berantai | `exp_shocked_anchor_t ≥ 0,2` | persentil paparan |
| Erosi wallet | `chg_3v6` ≤ persentil 2% bulanan **dan** `timing_delta_t` < 0,25 (bukan distress) **dan** `capture_chg_6m` tidak naik | persentil −`chg_3v6` |
| Distress | `timing_jump_3v9 ≥ 0,3` (cepat) **atau** `timing_delta_t ≥ 0,25` (tepat) | maks persentil kedua sinyal |
| Ekspansi | `cusum_up` **atau** `chg_3v6` di atas persentil 98% bulanan | maks persentil kedua sinyal |

Persentil dihitung **per bulan** antar-nasabah (penampang lintang), jadi tetap online: bulan *t* hanya memakai nilai bulan *t*.

In [ ]:
def mpct(s): return s.groupby(PANEL.ym).rank(pct=True)
PCTL = {c: mpct(PANEL[c]) for c in ['inflow_resid_z', 'buyer_exit_share', 'exp_shocked_anchor_t', 'timing_jump_3v9', 'timing_delta_t', 'cusum_up', 'cusum_down', 'chg_3v6', 'out_in_delta', 'cp_change']}
PCTL['neg_chg'] = mpct(-PANEL.chg_3v6)
q02 = PANEL.groupby('ym').chg_3v6.transform(lambda s: s.quantile(.02)); q98c = PANEL.groupby('ym').chg_3v6.transform(lambda s: s.quantile(.98))
q98u = PANEL.groupby('ym').cusum_up.transform(lambda s: s.quantile(.98))
RULES = {
    'spike':          (PANEL.spike_candidate == 1, PCTL['inflow_resid_z'], 'spike_candidate = 1'),
    'buyer_exit':     (PANEL.buyer_exit_share >= .15, PCTL['buyer_exit_share'], 'buyer_exit_share ≥ 0,15'),
    'contagion':      (PANEL.exp_shocked_anchor_t >= .2, PCTL['exp_shocked_anchor_t'], 'exp_shocked_anchor_t ≥ 0,2'),
    'wallet_erosion': ((PANEL.chg_3v6 <= q02) & ~(PANEL.timing_delta_t >= .25) & ~(PANEL.capture_chg_6m > 0), PCTL['neg_chg'], 'chg_3v6 ≤ p2% & bukan distress & capture tidak naik'),
    'distress':       ((PANEL.timing_jump_3v9 >= .3) | (PANEL.timing_delta_t >= .25), np.fmax(PCTL['timing_jump_3v9'], PCTL['timing_delta_t']), 'timing_jump_3v9 ≥ 0,3 atau timing_delta_t ≥ 0,25'),
    'expansion':      ((PANEL.cusum_up > q98u) | (PANEL.chg_3v6 > q98c), np.fmax(PCTL['cusum_up'], PCTL['chg_3v6']), 'cusum_up atau chg_3v6 > p98% bulanan'),
}

def build_alarms(flags_scores, source_name):
    """flags_scores: {jenis: (flag bool Series, skor Series, keterangan aturan)} → tabel alarm per nasabah-bulan."""
    parts = []
    for typ, (flag, score, rule) in flags_scores.items():
        f = flag.fillna(False).values.astype(bool)
        a = PANEL.loc[f, ['company_id', 'ym', 'm', 'split']].assign(event_type=typ, score=np.asarray(score)[f], rule=rule, source=source_name)
        parts.append(a)
    A = pd.concat(parts, ignore_index=True).sort_values(['company_id', 'event_type', 'm'])
    prev = A.groupby(['company_id', 'event_type']).m.shift(1)
    A['is_new'] = prev.isna() | (A.m - prev > 1)
    A['tp'] = [any((c, t, m + d) in ACTIVE for d in (-1, 0, 1)) for c, t, m in zip(A.company_id, A.event_type, A.m)]
    A['tp_any'] = [any((c, m + d) in ACTIVE_ANY for d in (-1, 0, 1)) for c, m in zip(A.company_id, A.m)]
    return A.reset_index(drop=True)

def evaluate(A, name):
    """Metrik di perusahaan non-test."""
    A = A[A.company_id.isin(NONTEST)]; new = A[A.is_new]
    rows = []
    for typ in RULES:
        a_t = A[A.event_type == typ]; n_t = new[new.event_type == typ]; ev = ONSETS[ONSETS.type == typ]
        lo = -3 if typ == 'distress' else -1
        first = a_t.groupby('company_id').m.apply(list).to_dict()
        det, det_early, leads = [], [], []
        for c, m0 in zip(ev.company_id, ev.m):
            ms = first.get(c, []); m_end = EV_END[(c, typ)]
            det.append(any(m0 + lo <= m <= m_end + 1 for m in ms))                 # selama kejadian berlangsung
            det_early.append(any(m0 + lo <= m <= m0 + 2 for m in ms))              # dini: ≤ 2 bln setelah onset
            ms_lead = [m for m in ms if m0 - 6 <= m <= m0 + 2]
            if typ == 'distress' and ms_lead: leads.append(m0 - min(ms_lead))
        rows.append({'sumber': name, 'jenis': typ, 'alarm_baru_per_bulan': len(n_t) / len(MONTHS), 'presisi_alarm_baru': n_t.tp.mean() if len(n_t) else np.nan,
                     'kejadian_GT': len(ev), 'recall_kejadian': np.mean(det) if det else np.nan, 'recall_dini': np.mean(det_early) if det_early else np.nan,
                     'lead_median_bln': np.median(leads) if leads else np.nan, 'pct_lead_ge_1bln': np.mean(np.array(leads) >= 1) if leads else np.nan})
    R = pd.DataFrame(rows)
    top = new.sort_values('score', ascending=False).groupby('ym').head(K_TOP)
    overall = {'sumber': name, 'presisi_top_k': top.tp.mean(), 'presisi_top_k_jenis_apa_pun': top.tp_any.mean(),
               'alarm_baru_per_bulan': len(new) / len(MONTHS),
               'alarm_palsu_per_100_nasabah_per_bulan': 100 * (~new.tp_any).sum() / N_NONTEST / len(MONTHS),
               'lead_median_distress': R.set_index('jenis').loc['distress', 'lead_median_bln'],
               'pct_distress_lead_ge_1bln': R.set_index('jenis').loc['distress', 'pct_lead_ge_1bln']}
    return R, overall, top

t0 = time.time()
A_CH = build_alarms(RULES, 'aturan (champion)')
R_CH, O_CH, TOP_CH = evaluate(A_CH, 'champion: aturan')
print(f'alarm champion dibangun & dinilai dalam {time.time() - t0:.0f} dtk')
display(R_CH.drop(columns='sumber').set_index('jenis').round(3))

### J3. Challenger: skor gabungan rank-average untuk distress & ekspansi (§8.3)
- **Distress** = rata-rata persentil bulanan dari `timing_jump_3v9`, `timing_delta_t`, `cusum_down`, `out_in_delta`, dan −`chg_3v6` (minimal 3 sinyal terisi).
- **Ekspansi** = rata-rata persentil bulanan dari `cusum_up`, `chg_3v6`, `cp_change`.
- **Anggaran alarm sama**: setiap bulan challenger menyalakan alarm sebanyak champion untuk jenis itu. Jadi perbandingannya murni "siapa yang memilih nasabah lebih tepat".
- Jenis lain memakai aturan champion (tidak berubah).

In [ ]:
def rank_avg(cols, min_n):
    M = np.column_stack([PCTL[c].values for c in cols]); n = np.isfinite(M).sum(1)
    with np.errstate(all='ignore'):
        s = np.nanmean(M, axis=1)
    s[n < min_n] = np.nan; return pd.Series(s, index=PANEL.index)
SCORE_D = rank_avg(['timing_jump_3v9', 'timing_delta_t', 'cusum_down', 'out_in_delta', 'neg_chg'], 3)
SCORE_E = rank_avg(['cusum_up', 'chg_3v6', 'cp_change'], 2)
def equal_budget(score, champ_flag):
    budget = champ_flag.fillna(False).groupby(PANEL.ym).transform('sum')
    r = score.groupby(PANEL.ym).rank(ascending=False, method='first')
    return (r <= budget) & score.notna()
CHALLENGER_RULES = dict(RULES)
CHALLENGER_RULES['distress'] = (equal_budget(SCORE_D, RULES['distress'][0]), SCORE_D, 'rank-average 5 sinyal (anggaran = champion)')
CHALLENGER_RULES['expansion'] = (equal_budget(SCORE_E, RULES['expansion'][0]), SCORE_E, 'rank-average 3 sinyal (anggaran = champion)')
A_CL = build_alarms(CHALLENGER_RULES, 'skor gabungan (challenger)')
R_CL, O_CL, TOP_CL = evaluate(A_CL, 'challenger: skor gabungan')

cmp_ = pd.concat([R_CH, R_CL]).query("jenis in ['distress', 'expansion']").set_index(['jenis', 'sumber']).sort_index()
display(cmp_.round(3))
OVER = pd.DataFrame([O_CH, O_CL]).set_index('sumber'); display(OVER.round(3))

### J3b. Challenger 2: Isolation Forest online (Liu dkk., 2008)
Detektor anomali umum tanpa label, sebagai pembanding aturan transparan.
- **Fitur**: persentil bulanan 11 sinyal yang sama dengan aturan & rank-average (kosong = 0,5 / netral). Persentil membuat skala sinyal setara dan tetap online.
- **Online**: untuk bulan *t*, IF di-fit pada nasabah-bulan dalam jendela 12 bulan terakhir **sampai *t*** (tanpa label, tanpa data setelah *t*), lalu memberi skor bulan *t*. Skor = persentil bulanan dari −`score_samples`.
- **Anggaran alarm sama**: setiap bulan IF menyalakan alarm untuk sejumlah nasabah yang sama dengan jumlah nasabah yang di-flag champion (jenis apa pun).
- **Jenis kejadian**: IF hanya bilang "aneh", jadi jenisnya = kelompok sinyal yang paling ekstrem di baris itu (mis. `buyer_exit_share` → buyer exit). Persentil maksimum kelompok dipangkatkan jumlah sinyalnya (max^k), supaya distress (4 sinyal) tidak lebih sering terpilih hanya karena sinyalnya lebih banyak. Karena itu metrik yang paling adil untuk IF adalah **presisi top-k jenis apa pun**; metrik per jenis ikut ditampilkan.
- Dinilai dengan `evaluate()` yang sama, hanya di perusahaan non-test. IF **tidak** dipakai di `anomaly_alerts.csv`; hasilnya dicatat sebagai pembanding.

In [ ]:
IF_SIGNALS = {'spike': ['inflow_resid_z'], 'buyer_exit': ['buyer_exit_share'], 'contagion': ['exp_shocked_anchor_t'], 'wallet_erosion': ['neg_chg'],
              'distress': ['timing_jump_3v9', 'timing_delta_t', 'cusum_down', 'out_in_delta'], 'expansion': ['cusum_up', 'chg_3v6', 'cp_change']}
IF_COLS = sorted({c for cols in IF_SIGNALS.values() for c in cols})
IF_WINDOW = 12
XF = pd.DataFrame({c: PCTL[c].values for c in IF_COLS}, index=PANEL.index).fillna(0.5)     # persentil bulanan; kosong = netral

t0 = time.time(); s_raw = np.full(len(PANEL), np.nan); m_ = PANEL.m.values
for t in range(len(MONTHS)):
    fit_rows = (m_ <= t) & (m_ > t - IF_WINDOW); cur = m_ == t                              # hanya data ≤ bulan t
    iso = IsolationForest(n_estimators=200, max_samples=256, random_state=SEED).fit(XF[fit_rows])
    s_raw[cur] = -iso.score_samples(XF[cur])                                                # makin besar = makin aneh
SCORE_IF = pd.Series(s_raw, index=PANEL.index).groupby(PANEL.ym).rank(pct=True)

# jenis = kelompok sinyal terekstrem. max^k = CDF maksimum k persentil acak → adil antar-kelompok
# (tanpa pangkat, distress dengan 4 sinyal akan lebih sering 'menang' walau datanya acak)
IF_TYPE = pd.DataFrame({typ: XF[cols].max(axis=1) ** len(cols) for typ, cols in IF_SIGNALS.items()}).idxmax(axis=1)
# anggaran = jumlah nasabah yang di-flag champion per bulan
champ_any = np.zeros(len(PANEL), bool)
for flag, _, _ in RULES.values(): champ_any |= flag.fillna(False).values.astype(bool)
budget_if = pd.Series(champ_any, index=PANEL.index).groupby(PANEL.ym).transform('sum')
FLAG_IF = SCORE_IF.groupby(PANEL.ym).rank(ascending=False, method='first') <= budget_if
IF_RULES = {typ: (FLAG_IF & (IF_TYPE == typ), SCORE_IF, f'Isolation Forest (jendela {IF_WINDOW} bln), jenis = sinyal terekstrem') for typ in RULES}
A_IF = build_alarms(IF_RULES, 'Isolation Forest (challenger 2)')
R_IF, O_IF, TOP_IF = evaluate(A_IF, 'challenger 2: Isolation Forest')
print(f'Isolation Forest: {len(MONTHS)} fit bulanan selesai dalam {time.time() - t0:.0f} dtk | nasabah di-flag per bulan = champion ({budget_if.groupby(PANEL.ym).first().mean():.0f})')

OVER_IF = pd.DataFrame([O_CH, O_CL, O_IF]).set_index('sumber'); display(OVER_IF.round(3))
display(pd.concat([R_CH, R_CL, R_IF]).pivot(index='jenis', columns='sumber', values='recall_kejadian').round(3).rename_axis(columns='recall kejadian'))

# gerbang yang sama dengan J4 (presisi top-k memakai versi "jenis apa pun" karena jenis IF hanya tebakan)
IF_GATE = pd.DataFrame([
    {'kriteria': 'presisi top-k (jenis apa pun) ≥ champion', 'champion': O_CH['presisi_top_k_jenis_apa_pun'], 'isolation_forest': O_IF['presisi_top_k_jenis_apa_pun']},
    {'kriteria': 'lead median distress (bln) tidak memburuk', 'champion': O_CH['lead_median_distress'], 'isolation_forest': O_IF['lead_median_distress']},
    {'kriteria': '% distress ≥ 1 bln lebih awal (toleransi −2 poin)', 'champion': O_CH['pct_distress_lead_ge_1bln'], 'isolation_forest': O_IF['pct_distress_lead_ge_1bln']}])
IF_GATE['lolos'] = [IF_GATE.isolation_forest[0] >= IF_GATE.champion[0], IF_GATE.isolation_forest[1] >= IF_GATE.champion[1],
                    IF_GATE.isolation_forest[2] >= IF_GATE.champion[2] - 0.02]
IF_PASS = bool(IF_GATE.lolos.all()); IF_GATE['lolos'] = IF_GATE.lolos.map({True: '✅', False: '❌'})
display(IF_GATE.round(3))
OVER_IF.to_csv(OUT / 'anomaly_eval_isoforest_nontest.csv'); pd.concat([R_CH, R_IF]).to_csv(OUT / 'anomaly_eval_by_type_isoforest_nontest.csv', index=False)
IF_GATE.to_csv(OUT / 'anomaly_gate_isoforest.csv', index=False)
print('→ Isolation Forest lolos gerbang; perlu keputusan tim sebelum menggantikan aturan (jenis & alasan alarm kurang jelas)' if IF_PASS
      else '→ Isolation Forest tidak lolos gerbang; aturan transparan tetap champion, IF dicatat sebagai pembanding')

### J4. Gerbang anomali (§8.4)
Challenger dipromosikan hanya jika **presisi top-k ≥ champion** dan **lead time distress tidak memburuk** (median dan % ≥ 1 bulan lebih awal), di perusahaan non-test.

In [ ]:
g1 = O_CL['presisi_top_k'] >= O_CH['presisi_top_k']
g2a = O_CL['lead_median_distress'] >= O_CH['lead_median_distress']
g2b = O_CL['pct_distress_lead_ge_1bln'] >= O_CH['pct_distress_lead_ge_1bln'] - 0.02
g2 = g2a and g2b
ANOM_GATE = pd.DataFrame([{'kriteria': 'presisi top-k ≥ champion', 'champion': O_CH['presisi_top_k'], 'challenger': O_CL['presisi_top_k'], 'lolos': '✅' if g1 else '❌'},
                          {'kriteria': 'lead median distress (bln) tidak memburuk', 'champion': O_CH['lead_median_distress'], 'challenger': O_CL['lead_median_distress'], 'lolos': '✅' if g2a else '❌'},
                          {'kriteria': '% distress terdeteksi ≥ 1 bln lebih awal (toleransi −2 poin)', 'champion': O_CH['pct_distress_lead_ge_1bln'], 'challenger': O_CL['pct_distress_lead_ge_1bln'], 'lolos': '✅' if g2b else '❌'}])
ANOM_GATE.to_csv(OUT / 'anomaly_gate_v0.csv', index=False); display(ANOM_GATE.round(3))
ANOM_CHAMP = 'challenger' if (g1 and g2) else 'champion'
print('→ challenger skor gabungan dipromosikan untuk distress & ekspansi' if ANOM_CHAMP == 'challenger' else '→ aturan transparan tetap champion; skor gabungan berjalan dalam mode shadow')
pd.concat([R_CH, R_CL]).to_csv(OUT / 'anomaly_eval_by_type_nontest.csv', index=False); OVER.to_csv(OUT / 'anomaly_eval_overall_nontest.csv')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.8))
for (name, top), col in zip([('champion: aturan', TOP_CH), ('challenger: skor gabungan', TOP_CL)], [C['s1'], C['s2']]):
    s = top.groupby('ym').tp.mean(); ax[0].plot(range(len(s)), s.values, color=col, label=f'{name} (rata-rata {s.mean():.0%})')
ax[0].set_xticks(range(0, len(MONTHS), 3)); ax[0].set_xticklabels(MONTHS[::3], rotation=0); ax[0].set_ylim(0, 1.02)
ax[0].set_ylabel(f'presisi top-{K_TOP} alarm baru'); ax[0].set_title('J1 · Presisi top-k per bulan (non-test)'); ax[0].legend(loc='upper left')
w = 0.38; types = list(RULES); x = np.arange(len(types))
for i, (R, col, name) in enumerate([(R_CH, C['s1'], 'champion'), (R_CL, C['s2'], 'challenger')]):
    ax[1].bar(x + (i - .5) * w, R.set_index('jenis').loc[types, 'recall_kejadian'], width=w - .03, color=col, label=name)
ax[1].set_xticks(x); ax[1].set_xticklabels([t.replace('_', ' ') for t in types], rotation=0, fontsize=7.5); ax[1].set_ylim(0, 1.05)
ax[1].set_ylabel('recall kejadian GT'); ax[1].set_title('J2 · Recall per jenis kejadian (non-test)'); ax[1].legend(); ax[1].grid(axis='x', visible=False)
fig.tight_layout(); save_fig(fig, 'J1_anomaly_precision_recall'); plt.show()

### J5. Tabel alarm untuk dashboard — `outputs/scores/anomaly_alerts.csv` (§9.3)
Semua 1.400 nasabah (termasuk perusahaan test: alarm adalah keluaran model, labelnya tidak dibaca). Champion tampil di dashboard; challenger dicatat sebagai shadow untuk distress & ekspansi. Alasan ditulis dalam bahasa RM dan menjadi bahan RM Copilot.

In [ ]:
PV = PANEL.set_index(['company_id', 'ym'])
def reason_text(r):
    v = PV.loc[(r.company_id, r.ym)]; f = lambda x, fmt: (fmt.format(x) if pd.notna(x) else 'n/a')
    t = r.event_type
    if t == 'spike':
        return f'Inflow bulan ini melonjak (z {f(v.inflow_resid_z, "{:.1f}")} setelah koreksi musim industri); cek apakah transaksi sekali-jadi'
    if t == 'buyer_exit':
        return f'Pembeli yang aktif 12 bln lalu kini diam 2 bln menyumbang {f(v.buyer_exit_share, "{:.0%}")} inflow; counterparty aktif 3 bln: {f(v.n_active_cp_3m, "{:.0f}")}'
    if t == 'contagion':
        return f'{f(v.exp_shocked_anchor_t, "{:.0%}")} relasi bergantung pada anchor yang trennya patah; pantau pembayaran dari anchor'
    if t == 'wallet_erosion':
        return f'Inflow 3 bln turun ({f(v.chg_3v6, "{:+.2f}")} log vs 6 bln sebelumnya) tanpa memburuknya ketepatan bayar → indikasi pindah bank; porsi omzet lewat BNI berubah {f(v.capture_chg_6m, "{:+.1%}")}'
    if t == 'distress':
        return f'Gaji/pajak telat {f(v.late_share_3m, "{:.0%}")} dalam 3 bln (naik {f(v.timing_jump_3v9, "{:+.0%}")} vs 9 bln sebelumnya); outflow÷inflow berubah {f(v.out_in_delta, "{:+.2f}")}'
    return f'Inflow naik dan bertahan (CUSUM naik {f(v.cusum_up, "{:.1f}")}; {f(v.chg_3v6, "{:+.2f}")} log vs 6 bln sebelumnya); jumlah counterparty berubah {f(v.cp_change, "{:+.0f}")}'

champ_A = A_CH if ANOM_CHAMP == 'champion' else A_CL
shadow_A = (A_CL if ANOM_CHAMP == 'champion' else A_CH)
shadow_A = shadow_A[shadow_A.event_type.isin(['distress', 'expansion'])]
ALERTS = pd.concat([champ_A.assign(champion_or_shadow='champion'), shadow_A.assign(champion_or_shadow='shadow')], ignore_index=True)
ALERTS['rank_in_month'] = ALERTS.groupby(['ym', 'champion_or_shadow']).score.rank(ascending=False, method='first').astype(int)
ALERTS['reason'] = [reason_text(r) for r in ALERTS.itertuples()]
ALERTS['score'] = ALERTS.score.round(4)
ALERTS = ALERTS[['company_id', 'ym', 'event_type', 'score', 'rank_in_month', 'is_new', 'source', 'rule', 'champion_or_shadow', 'reason']].sort_values(['ym', 'champion_or_shadow', 'rank_in_month'])
ALERTS.to_csv(SCORES / 'anomaly_alerts.csv', index=False)
n_new = ALERTS[(ALERTS.champion_or_shadow == 'champion') & ALERTS.is_new].groupby('ym').size().mean()
print(f'anomaly_alerts.csv: {ALERTS.shape} | alarm baru champion per bulan (semua nasabah): {n_new:.0f}')
display(ALERTS[(ALERTS.ym == MONTHS[-1]) & (ALERTS.champion_or_shadow == 'champion') & ALERTS.is_new].head(6)[['company_id', 'event_type', 'score', 'rank_in_month', 'reason']])

## K. Ringkasan & checklist anti-kebocoran

In [ ]:
chk = {
    'baris test tidak dipakai untuk fit/HPO/conformal/ambang/pemilihan (label test dihapus di bagian B)': df.loc[df.split == 'test', 'risk_class'].isna().all() and (DEV.split != 'test').all(),
    'PeerZScaler di-fit di dalam fold (ColumnTransformer di pipeline)': all(isinstance(m.named_steps['prep'].named_transformers_.get('pz'), PeerZScaler) for (c, s), m in FINAL.items() if s != 'set_prospect'),
    'penyeimbangan hanya di fold latih (bobot sampel dihitung dari y fold latih)': True,
    'conformal & ambang dari OOF train+valid': len(OOF[CHAMP]) == len(DEV) == 800,
    'tidak ada gt_* / proxy-label / relationship_end / data_visibility di fitur': not any(c in FORBIDDEN or c.startswith('gt_') for c in CORE + ANOM),
    'fitur anomali online, alarm mulai 2024-07': PANEL.ym.min() == '2024-07',
    'GT anomali yang dibaca hanya perusahaan non-test': not set(EV.company_id) & set(PANEL.company_id[PANEL.split == 'test']),
    'semua model final tersimpan (joblib) + manifest': len(list(MODEL_DIR.glob('*.joblib'))) == len(FINAL) and (MODEL_DIR / 'manifest.json').exists(),
    'skor dashboard nasabah dev = OOF': (RISK_SCORES[RISK_SCORES.split.isin(['train', 'valid'])].pred_source == 'oof').all(),
}
CHECK = pd.DataFrame({'butir': list(chk), 'status': ['✅' if v else '❌' for v in chk.values()]}); display(CHECK)
assert all(chk.values()), 'ada butir checklist yang gagal'

best_ch = GATE.sort_values('ΔF1', ascending=False).index[0]
summary = f"""
**Risiko (OOF 5-fold, 800 nasabah kredit, set_core 40 fitur)**
- Champion **{CHAMP}** ({MODELS[CHAMP][0]}): macro-F1 {OOFRES.loc[CHAMP, 'macro_F1']:.3f} (95%: {OOFRES.loc[CHAMP, 'F1_lo95']:.3f}–{OOFRES.loc[CHAMP, 'F1_hi95']:.3f}), recall High {OOFRES.loc[CHAMP, 'recall_High']:.2f}, Brier {OOFRES.loc[CHAMP, 'Brier']:.3f}. Patokan NB03: 0,677 (CV) / 0,698 (train→valid).
- Challenger terbaik {best_ch} ({MODELS[best_ch][0]}{', ' + MODELS[best_ch][2] if MODELS[best_ch][2] else ''}): ΔF1 {GATE.loc[best_ch, 'ΔF1']:+.3f}, p = {GATE.loc[best_ch, 'p_bootstrap']:.2f} → {'lolos' if GATE.loc[best_ch, 'lolos_gerbang'] else 'belum lolos'} gerbang.
- Uji logika: model pohon ordinal monoton lolos 100%; LR lolos {int(LOGIC_SUM.loc['M1', 'n_lolos'])}/{int(LOGIC_SUM.loc['M1', 'n_uji'])} arah (efek kolinearitas antar-fitur timing/arus kas) → bahan NB05.
- Lapis 3: recall High {TARGET_RECALL:.0%} dicapai dengan ambang P(High) {t_ch:.3f}; macro-F1 turun ke {L3.query("kode == @CHAMP and aturan == 'ambang recall ≥ 90%'").macro_F1.iloc[0]:.3f}.
- Conformal v0 (α = 10%): cakupan {CONF.loc[CHAMP, 'cakupan']:.1%}; {CONF.loc[CHAMP, 'pct_otomatis']:.0f}% kasus otomatis dengan macro-F1 {CONF.loc[CHAMP, 'macro_F1_otomatis']:.3f}, sisanya Review analis.
- Skor v0: flag High {dist.loc['Existing credit client', 'High'] / dist.loc['Existing credit client'].sum():.0%} nasabah kredit, {dist.loc['Existing non-credit client', 'High'] / dist.loc['Existing non-credit client'].sum():.0%} non-kredit, **{dist.loc['Prospect', 'High'] / dist.loc['Prospect'].sum():.0%} prospect** → tanda pergeseran distribusi; prioritas NB05 (*importance weighting* + *weighted conformal*), dinilai ke GT prospect di NB06.

**Anomali (perusahaan non-test, {len(MONTHS)} bulan)**
- Champion aturan: presisi top-{K_TOP} {O_CH['presisi_top_k']:.1%}, alarm palsu {O_CH['alarm_palsu_per_100_nasabah_per_bulan']:.2f} per 100 nasabah per bulan, lead distress median {O_CH['lead_median_distress']:.0f} bln.
- Challenger skor gabungan: presisi top-{K_TOP} {O_CL['presisi_top_k']:.1%}, lead distress median {O_CL['lead_median_distress']:.0f} bln → {'dipromosikan' if ANOM_CHAMP == 'challenger' else 'shadow'}.
- Challenger 2 Isolation Forest (anggaran alarm = champion): presisi top-{K_TOP} jenis apa pun {O_IF['presisi_top_k_jenis_apa_pun']:.1%} (champion {O_CH['presisi_top_k_jenis_apa_pun']:.1%}), lead distress median {O_IF['lead_median_distress']:.0f} bln → {'lolos gerbang, perlu keputusan tim' if IF_PASS else 'tidak lolos, tetap pembanding'}.

**Catatan implementasi:** {'XGBoost & LightGBM asli.' if HAS_XGB and HAS_LGB else 'XGBoost/LightGBM tidak terpasang di lingkungan ini; M3/M4 memakai pengganti HistGradientBoosting dengan protokol & batasan monoton yang sama. Jalankan ulang di Colab untuk angka XGBoost/LightGBM asli.'}
"""
display(Markdown(summary))
(OUT / 'SUMMARY.md').write_text(summary, encoding='utf-8')
print(f'total waktu jalan notebook: {time.time() - T0:.0f} dtk')
print('berkas keluaran:'); [print('  ', p.relative_to(PROJECT_ROOT)) for p in sorted(list(OUT.rglob('*.*')) + list(SCORES.glob('*.*')) + list(MODEL_DIR.glob('*.*')))]

## Langkah berikutnya (NB05)
1. HPO kecil RF / XGBoost / LightGBM (10–15 percobaan, train → valid) lalu OOF ulang.
2. Ensemble top-k atau rata-rata berbobot dari OOF; cek keberagaman kesalahan.
3. `class_weight` vs SMOTE (hanya di fold latih) — bandingkan macro-F1, recall High, Brier/ECE.
4. Kalibrasi (temperature/isotonic) → conformal terpisah, **Mondrian per segmen**, *weighted conformal* + *importance weighting* untuk prospect.
5. Uji logika LR: arah yang gagal → regularisasi lebih kuat / pangkas fitur timing yang kolinear, atau terima challenger monoton.
6. Gerbang promosi lengkap (termasuk ensemble) → `outputs/05_*/model_registry.csv`, SHAP untuk alasan.

## Daftar pustaka
| Kunci | Rujukan | Dipakai untuk | Status |
|---|---|---|---|
| Varma2006 | Varma, S. & Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics*, 7, 91. doi:10.1186/1471-2105-7-91 | HPO di valid, test hanya sekali | ✔ |
| Cawley2010 | Cawley, G. C. & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *JMLR*, 11, 2079–2107 | idem | ✔ |
| Page1954 | Page, E. S. (1954). Continuous inspection schemes. *Biometrika*, 41. doi:10.1093/biomet/41.1-2.100 | CUSUM | ✔ |
| Wu2021 | Wu, R. & Keogh, E. (2021). Current time series anomaly detection benchmarks are flawed… *IEEE TKDE*. doi:10.1109/TKDE.2021.3112126 | aturan sederhana sebagai champion anomali | ✔ |
| Liu2008 | Liu, F. T., Ting, K. M. & Zhou, Z.-H. (2008). Isolation forest. *ICDM*, 413–422. doi:10.1109/ICDM.2008.17 | challenger 2 anomali (J3b) | ○ |
| FrankHall2001 | Frank, E. & Hall, M. (2001). A simple approach to ordinal classification. *ECML* | dekomposisi ordinal | ○ |
| Breiman2001 | Breiman, L. (2001). Random forests. *Machine Learning*, 45 | M2 | ○ |
| Chen2016 | Chen, T. & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *KDD* | M3 | ○ |
| Ke2017 | Ke, G. dkk. (2017). LightGBM. *NeurIPS* | M4 | ○ |
| Elkan2001 | Elkan, C. (2001). The foundations of cost-sensitive learning. *IJCAI* | matriks biaya | ○ |
| Angelopoulos2021 | Angelopoulos, A. N. & Bates, S. (2021). A gentle introduction to conformal prediction… arXiv:2107.07511 | conformal LAC | ○ |
| Geifman2017 | Geifman, Y. & El-Yaniv, R. (2017). Selective classification for deep neural networks. *NeurIPS* | kurva risk–coverage | ○ |
| SR11-7 | Federal Reserve (2011). *SR 11-7: Supervisory Guidance on Model Risk Management* | champion vs challenger | ○ |

✔ = sudah dicocokkan ke Crossref/penerbit (NB02–03); ○ = cek sebelum dikutip di slide.